# Python Basics for Data Engineering Interviews

This notebook is a hands-on tour of the Python language features that come up in data-engineering interviews:
classes and methods, dunder methods, exceptions, context managers, generators, decorators, typing, and the
concurrency basics everyone gets asked about.

**How to use it**

1. Run the cells top to bottom (`Shift+Enter`).
2. Every concept has a runnable example and most have a **self-check** cell with `assert`s, so you know immediately
   whether your mental model is right.
3. The last section is a bank of interview questions with model answers.

Companion notebooks: `02_python_coding_problems.ipynb` (algorithms), `03_sql_interview_questions.ipynb` (SQL),
`04_pyspark_concepts.ipynb` (Spark), `05_interview_questions.ipynb` (mixed question bank).

## 1. Values, types and the four workhorse collections

Python is dynamically typed: names are bound to objects, and the object knows its type. The four collections you
will use constantly are `list` (ordered, mutable), `tuple` (ordered, immutable), `set` (unique, unordered) and
`dict` (key-value, insertion-ordered since 3.7).

In [1]:
numbers = [3, 1, 4, 1, 5, 9, 2, 6]
print("list:", numbers, "| sorted:", sorted(numbers), "| unique:", sorted(set(numbers)))

record = ("order-1", 42.5)                 # tuple: fixed shape
name, amount = record                      # unpacking
print("tuple:", record, name, amount)

squares = {n: n * n for n in range(5)}     # dict comprehension
evens = [n for n in numbers if n % 2 == 0] # list comprehension
print("dict:", squares)
print("evens:", evens)

text = "  Data Engineering  "
print("strip/lower/split:", text.strip().lower().split())
print("f-string:", f"{len(numbers)} values, max={max(numbers)}, total={sum(numbers)}")
print("slice:", numbers[2:5], "| step:", numbers[::2])

list: [3, 1, 4, 1, 5, 9, 2, 6] | sorted: [1, 1, 2, 3, 4, 5, 6, 9] | unique: [1, 2, 3, 4, 5, 6, 9]
tuple: ('order-1', 42.5) order-1 42.5
dict: {0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
evens: [4, 2, 6]
strip/lower/split: ['data', 'engineering']
f-string: 8 values, max=9, total=31
slice: [4, 1, 5] | step: [3, 4, 5, 2]


**Exercise 1.** Write `top_n_by_value(counts, n)` that returns the `n` keys with the highest counts, breaking ties
by key name so the result is deterministic.

In [2]:
def top_n_by_value(counts: dict, n: int) -> list:
    # sort by count descending, then key ascending
    return sorted(counts, key=lambda k: (-counts[k], k))[:n]

assert top_n_by_value({"b": 2, "a": 2, "c": 5}, 2) == ["c", "a"]
assert top_n_by_value({}, 3) == []
assert top_n_by_value({"x": 1}, 5) == ["x"]
print("exercise 1 passed")

exercise 1 passed


## 2. Mutability, references and copies

Assignment binds a name to an object; it does not copy. Functions receive references, so mutating a list argument
is visible to the caller. Interviewers love this because it explains surprise bugs in pipeline code.

In [3]:
a = [1, 2, 3]
b = a                    # same object
b.append(4)
assert a == [1, 2, 3, 4] and a is b

import copy
c = a.copy()             # shallow copy
c.append(99)
assert a == [1, 2, 3, 4] and c == [1, 2, 3, 4, 99]

nested = [[1], [2]]
shallow = copy.copy(nested)
deep = copy.deepcopy(nested)
nested[0].append("boom")
assert shallow[0] == [1, "boom"]   # shallow shares the inner lists
assert deep[0] == [1]              # deep is fully independent
print("mutability checks passed")

mutability checks passed


## 3. Functions, scope and closures

Keyword arguments, `*args` / `**kwargs`, closures and the LEGB rule (Local, Enclosing, Global, Built-in) are
standard interview material.

In [4]:
def load(table, *, mode="append", **options):
    "mode is keyword-only; options collects anything else"
    return {"table": table, "mode": mode, **options}

assert load("orders", mode="overwrite", compression="zstd") == {
    "table": "orders", "mode": "overwrite", "compression": "zstd"
}

def make_counter():
    count = 0
    def increment():
        nonlocal count          # rebind the enclosing variable
        count += 1
        return count
    return increment

counter = make_counter()
assert [counter(), counter(), counter()] == [1, 2, 3]

def total(*values):             # variable positional args
    return sum(values)

assert total(1, 2, 3) == 6 and total() == 0
print("scope/closures checks passed")

scope/closures checks passed


### Pitfall: mutable default arguments and late binding

Two classics that show up in production code review questions.

In [5]:
def append_bad(item, bucket=[]):        # the default list is created once
    bucket.append(item)
    return bucket

assert append_bad("a") == ["a"]
assert append_bad("b") == ["a", "b"]    # surprising, but that is the point

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

assert append_good("a") == ["a"]
assert append_good("b") == ["b"]

callbacks = [lambda: i for i in range(3)]
assert [cb() for cb in callbacks] == [2, 2, 2]          # late binding
callbacks = [lambda i=i: i for i in range(3)]
assert [cb() for cb in callbacks] == [0, 1, 2]          # bound at definition
print("pitfall checks passed")

pitfall checks passed


## 4. Classes, objects and the three kinds of methods

* `__init__` initialises an instance; `self` is the instance.
* **Instance methods** take `self` and work on one object.
* **Class methods** take `cls`, often used as alternative constructors.
* **Static methods** take neither: they are namespaced helpers.

In [6]:
class Account:
    currency = "USD"                     # class attribute, shared

    def __init__(self, owner, balance=0.0):
        self.owner = owner               # instance attribute
        self.balance = balance           # per-object
        self.history = []

    def deposit(self, amount):           # instance method
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount
        self.history.append(("deposit", amount))
        return self.balance

    @classmethod
    def from_statement(cls, owner, amounts):    # alternative constructor
        account = cls(owner)
        for amount in amounts:
            account.deposit(amount)
        return account

    @staticmethod
    def fee_for(amount):                  # pure helper
        return round(amount * 0.01, 2)

    def __repr__(self):
        return f"Account(owner={self.owner!r}, balance={self.balance:.2f})"

account = Account.from_statement("ada", [100, 50])
assert account.balance == 150
assert Account.fee_for(200) == 2.0
assert Account.currency == "USD"
assert "balance=150.00" in repr(account)
print("class checks passed:", account)

class checks passed: Account(owner='ada', balance=150.00)


### Inheritance and polymorphism

`super()` calls the parent implementation; method resolution order (MRO) decides which class wins in a diamond.

In [7]:
class Storage:
    def __init__(self, path):
        self.path = path
    def save(self, rows):
        return f"wrote {len(rows)} rows to {self.path}"

class ParquetStorage(Storage):
    def save(self, rows):
        base = super().save(rows)                 # reuse parent behaviour
        return f"{base} as Parquet"

class DeltaStorage(Storage):
    def save(self, rows):
        return f"merged {len(rows)} rows into {self.path} (Delta)"

for backend in (ParquetStorage("s3://lake/curated"), DeltaStorage("s3://lake/delta")):
    print(backend.save([1, 2, 3]))

assert ParquetStorage("p").save([1]) == "wrote 1 rows to p as Parquet"
assert [c.__name__ for c in ParquetStorage.__mro__] == ["ParquetStorage", "Storage", "object"]

wrote 3 rows to s3://lake/curated as Parquet
merged 3 rows into s3://lake/delta (Delta)


## 5. Dunder methods: making your objects behave like built-ins

Defining `__repr__`, `__eq__`, `__hash__`, `__add__`, `__len__` and `__iter__` is the difference between a data bag
and a first-class value.

In [8]:
class Money:
    def __init__(self, cents, currency="USD"):
        self.cents = int(cents)
        self.currency = currency

    def __repr__(self):
        return f"Money({self.cents}, {self.currency!r})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return (self.cents, self.currency) == (other.cents, other.currency)

    def __hash__(self):
        return hash((self.cents, self.currency))     # keep eq/hash consistent

    def __add__(self, other):
        if self.currency != other.currency:
            raise ValueError("cannot add different currencies")
        return Money(self.cents + other.cents, self.currency)

    def __bool__(self):
        return self.cents != 0

price = Money(1250)
tax = Money(100)
assert price + tax == Money(1350)
assert len({Money(1), Money(1), Money(2)}) == 2     # hashable in a set
assert bool(Money(0)) is False and bool(price) is True
print("dunder checks passed:", price + tax)

dunder checks passed: Money(1350, 'USD')


## 6. Properties: validation without breaking the attribute API

`@property` turns a method into a computed attribute; the setter validates writes, so callers keep using
`obj.value = ...`.

In [9]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius          # goes through the setter

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self):               # read-only computed attribute
        return self._celsius * 9 / 5 + 32

t = Temperature(25)
assert t.fahrenheit == 77.0
t.celsius = -40
assert t.fahrenheit == -40.0
try:
    Temperature(-300)
except ValueError as exc:
    assert "absolute zero" in str(exc)
print("property checks passed")

property checks passed


## 7. Dataclasses and enums

`@dataclass` generates `__init__`, `__repr__` and `__eq__`; `frozen=True` makes it hashable/immutable.
`Enum` replaces magic strings.

In [10]:
from dataclasses import dataclass, field
from enum import Enum

class Status(Enum):
    PLACED = "placed"
    SHIPPED = "shipped"
    CANCELLED = "cancelled"

@dataclass(frozen=True)
class Order:
    order_id: str
    amount: float
    status: Status = Status.PLACED
    tags: tuple = field(default_factory=tuple)

@dataclass
class Cart:
    items: list = field(default_factory=list)     # fresh list per instance

    def add(self, item):
        self.items.append(item)
        return self

order = Order("o-1", 42.5, tags=("priority",))
assert order.status is Status.PLACED
assert order == Order("o-1", 42.5, tags=("priority",))
assert len({order, Order("o-1", 42.5, tags=("priority",))}) == 1   # frozen => hashable

c1, c2 = Cart(), Cart()
c1.add("a")
assert c1.items == ["a"] and c2.items == []      # no shared default
print("dataclass/enum checks passed")

dataclass/enum checks passed


## 8. Exceptions: hierarchy, `else`, `finally`, chaining

Catch the narrowest exception you can handle, add context with `raise ... from`, and use `else`/`finally` for the
success and cleanup paths.

In [11]:
class DataQualityError(Exception):
    "Base class for pipeline data-quality failures"

class EmptyPartitionError(DataQualityError):
    def __init__(self, partition):
        super().__init__(f"partition {partition} is empty")
        self.partition = partition

def validate(rows, partition="ds=2026-10-04"):
    if rows is None:
        raise ValueError("rows is None")
    if not rows:
        raise EmptyPartitionError(partition)
    return len(rows)

steps = []
try:
    steps.append(validate([], "ds=2026-10-03"))
except EmptyPartitionError as exc:
    steps.append(f"caught {exc.partition}")
else:
    steps.append("no error")
finally:
    steps.append("cleanup")

assert steps == ["caught ds=2026-10-03", "cleanup"]
assert isinstance(EmptyPartitionError("x"), DataQualityError)

try:
    validate(None)
except ValueError as exc:
    chained = exc
assert isinstance(chained, ValueError) and "rows is None" in str(chained)
print("exception checks passed")

exception checks passed


## 9. Context managers: guaranteed setup and teardown

Any object with `__enter__` / `__exit__` works with `with`. `contextlib.contextmanager` turns a generator into one.
Typical data-engineering uses: connections, transactions, temporary files, timers.

In [12]:
from contextlib import contextmanager
import time

class Transaction:
    def __init__(self, name):
        self.name = name
        self.log = []
    def __enter__(self):
        self.log.append(f"begin {self.name}")
        return self
    def __exit__(self, exc_type, exc, tb):
        self.log.append("rollback" if exc_type else f"commit {self.name}")
        return False            # do not swallow exceptions

tx = Transaction("load-orders")
with tx:
    tx.log.append("insert rows")
assert tx.log == ["begin load-orders", "insert rows", "commit load-orders"]

failing = Transaction("bad-load")
try:
    with failing:
        raise RuntimeError("warehouse down")
except RuntimeError:
    pass
assert failing.log[-1] == "rollback"

@contextmanager
def timer():
    started = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - started
        assert elapsed >= 0

with timer():
    pass
print("context manager checks passed")

context manager checks passed


## 10. Iterators and generators: memory-safe pipelines

A generator yields values lazily, so you can process a file larger than memory. Interviewers frequently ask you to
turn a list-based function into a generator.

In [13]:
def read_batches(rows, size):
    for start in range(0, len(rows), size):
        yield rows[start:start + size]

def parse_orders(lines):
    for line in lines:
        order_id, amount = line.split(",")
        yield {"order_id": order_id, "amount": float(amount)}

def only_positive(orders):
    for order in orders:
        if order["amount"] > 0:
            yield order

raw = ["o1,10.5", "o2,-3", "o3,7"]
parsed = list(parse_orders(raw))
filtered = list(only_positive(parse_orders(raw)))
assert parsed[0] == {"order_id": "o1", "amount": 10.5}
assert [o["order_id"] for o in filtered] == ["o1", "o3"]
assert list(read_batches([1, 2, 3, 4, 5], 2)) == [[1, 2], [3, 4], [5]]

# lazy: nothing is consumed until the generator is iterated
consumed = []
def tracked():
    for i in range(3):
        consumed.append(i)
        yield i

lazy = tracked()
assert consumed == []
assert list(lazy) == [0, 1, 2] and consumed == [0, 1, 2]
print("generator checks passed")

generator checks passed


## 11. Decorators: cross-cutting behaviour

A decorator is a function that takes a function and returns a replacement. `functools.wraps` preserves metadata;
`lru_cache` memoises; retry wrappers belong in every ingestion pipeline.

In [14]:
import functools

def traced(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

def retry(times=3):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except ConnectionError:
                    wrapper.attempts = attempt
                    if attempt == times:
                        raise
        return wrapper
    return decorator

@traced
def add(a, b):
    return a + b

assert add(1, 2) == 3 and add.calls == 1
assert add.__name__ == "add"            # functools.wraps preserved the name

state = {"fails": 2}
@retry(times=5)
def flaky():
    if state["fails"] > 0:
        state["fails"] -= 1
        raise ConnectionError("temporary")
    return "ok"

assert flaky() == "ok"

@functools.lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

assert fib(30) == 832040
assert fib.cache_info().misses <= 31
print("decorator checks passed")

decorator checks passed


## 12. Type hints

Hints document intent and let `mypy`/IDEs catch mistakes; they are not enforced at runtime. Use them on public
functions and data models.

In [15]:
from typing import Iterable, Iterator, Optional

def average(values: Iterable[float]) -> Optional[float]:
    total = 0.0
    count = 0
    for value in values:
        total += value
        count += 1
    return total / count if count else None

def running_total(values: Iterable[float]) -> Iterator[float]:
    total = 0.0
    for value in values:
        total += value
        yield total

def find_partition(partitions: dict[str, str], prefix: str) -> str | None:
    return partitions.get(prefix)

assert average([1, 2, 3]) == 2.0
assert average([]) is None
assert list(running_total([1, 2, 3])) == [1.0, 3.0, 6.0]
assert find_partition({"a": "s3://a"}, "a") == "s3://a"
print("typing checks passed")

typing checks passed


## 13. Dates, times and timezones

Store timestamps in UTC, attach a timezone, and convert only for display. `datetime.utcnow()` is deprecated; use
`datetime.now(timezone.utc)`.

In [16]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo

started = datetime(2026, 10, 4, 12, 0, tzinfo=timezone.utc)
finished = started + timedelta(hours=2, minutes=30)
assert (finished - started).total_seconds() == 9000

as_ny = finished.astimezone(ZoneInfo("America/New_York"))
assert as_ny.hour == 10 and as_ny.utcoffset() == timedelta(hours=-4)   # EDT

parsed = datetime.fromisoformat("2026-10-04T12:00:00+00:00")
assert parsed.tzinfo is not None and parsed == started

print("utc:", started.isoformat(), "| new york:", as_ny.isoformat())

utc: 2026-10-04T12:00:00+00:00 | new york: 2026-10-04T10:30:00-04:00


## 14. Files, `pathlib` and JSON

`pathlib.Path` is the modern file API; `json` and `csv` are stdlib. The examples write to a temporary directory so
nothing is left behind.

In [17]:
import json, tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    orders = [{"order_id": "o1", "amount": 10.5}, {"order_id": "o2", "amount": 7.0}]

    ndjson = root / "orders.ndjson"
    with ndjson.open("w", encoding="utf-8") as handle:
        for order in orders:
            handle.write(json.dumps(order) + "\n")

    loaded = [json.loads(line) for line in ndjson.read_text(encoding="utf-8").splitlines()]
    assert loaded == orders
    assert ndjson.suffix == ".ndjson" and ndjson.name == "orders.ndjson"

    total = sum(order["amount"] for order in loaded)
    assert total == 17.5
    print("round-tripped", len(loaded), "orders, total", total)

round-tripped 2 orders, total 17.5


## 15. Concurrency basics: GIL, threads, processes, asyncio

* The **GIL** lets only one thread run Python bytecode at a time, so threads help **I/O-bound** work, not CPU-bound
  work.
* Use **processes** (`ProcessPoolExecutor`) to bypass the GIL for CPU-bound work.
* Use **asyncio** for many concurrent network calls in one thread.

In [18]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch_partition(partition):
    time.sleep(0.01)                    # pretend this is a network call
    return f"rows for {partition}"

partitions = [f"ds=2026-10-{day:02d}" for day in range(1, 6)]
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(fetch_partition, partitions))
elapsed = time.perf_counter() - started

assert results[0] == "rows for ds=2026-10-01" and len(results) == 5
print(f"5 mocked fetches in {elapsed*1000:.1f} ms (sequential would be ~50 ms)")

5 mocked fetches in 11.0 ms (sequential would be ~50 ms)


An `asyncio` version of the same idea (shown as code to read, not run, because a notebook already runs an event
loop):

```python
import asyncio

async def fetch(partition):
    await asyncio.sleep(0.01)
    return f"rows for {partition}"

async def main():
    return await asyncio.gather(*(fetch(p) for p in partitions))
```

Rule of thumb for interviews: threads for blocking I/O with libraries that are not async-aware; asyncio when the
whole call chain is async; processes for CPU-heavy Python; and Spark for genuinely large-scale parallelism.

## 16. Common pitfalls checklist

Mutable defaults, late binding, float equality, string concatenation in loops, shadowing builtins, and `is` vs `==`.

In [19]:
# float comparison: use math.isclose, not ==
assert 0.1 + 0.2 != 0.3
import math
assert math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)

# build strings with join, not += in a loop
parts = [str(i) for i in range(5)]
assert "-".join(parts) == "0-1-2-3-4"

# is compares identity; == compares value
a, b = [1, 2], [1, 2]
assert a == b and a is not b

# do not shadow builtins
list_of_rows = [1, 2]        # not `list`
assert sum(list_of_rows) == 3
print("pitfall checks passed")

pitfall checks passed


## Interview questions

Answer these out loud before reading the model answers.

1. **What is the difference between a list and a tuple, and when do you prefer each?**
   Lists are mutable and used for homogeneous sequences you will grow or change; tuples are immutable, hashable and
   used for fixed-shape records (composite keys, function returns) where immutability is a feature.
2. **Explain the mutable default argument trap.**
   Default values are evaluated once at function definition, so a `def f(x, bucket=[])` shares one list across all
   calls. Use `None` and create the container inside the function.
3. **What is the difference between `__str__` and `__repr__`?**
   `__repr__` is the unambiguous developer representation (ideally `eval`-able) used in debugging; `__str__` is the
   human-friendly display used by `print`/f-strings. If only one exists, implement `__repr__`.
4. **Why must `__eq__` and `__hash__` be consistent?**
   Equal objects must hash equally or dict/set lookups break. If you define `__eq__` and objects should stay
   hashable, define `__hash__` (or use `@dataclass(frozen=True)`).
5. **When would you use `@classmethod` over `@staticmethod`?**
   `@classmethod` receives `cls` and is the idiomatic alternative constructor (`from_statement`, `from_config`);
   `@staticmethod` is a namespaced helper with no access to instance or class state.
6. **How do generators reduce memory usage?**
   They produce values lazily, keeping only the current item (plus generator state) in memory instead of materialising
   the whole sequence. This is how you stream a file larger than RAM.
7. **What does a context manager guarantee?**
   That teardown runs even when the block raises, which is why connections, transactions, locks and temp files use
   `with`. `contextlib.contextmanager` lets you write one from a generator.
8. **What does `functools.wraps` do, and why does it matter?**
   It copies `__name__`, `__doc__`, annotations and `__wrapped__` from the wrapped function so tooling, logs and
   introspection still see the original function.
9. **Explain the GIL and its practical consequence.**
   Only one thread executes Python bytecode at a time, so threads speed up I/O-bound work but not CPU-bound work;
   for CPU work use `ProcessPoolExecutor`, C extensions that release the GIL, or a distributed engine.
10. **`is` vs `==`?**
    `is` tests identity (same object, e.g. `x is None`); `==` tests value equality via `__eq__`.
11. **Why are type hints not enforcement?**
    They are metadata for static analysis and IDEs; nothing stops you passing a `str` where `int` is annotated unless
    you add runtime validation (pydantic, `isinstance` checks, or tests).
12. **How do you make a class immutable?**
    `@dataclass(frozen=True)`, `NamedTuple`, or define `__setattr__`/properties with no setter; immutability makes
    instances safely hashable and shareable across threads and processes.

## Keep practising

* `02_python_coding_problems.ipynb` — timed coding problems with self-checking tests.
* `03_sql_interview_questions.ipynb` — SQL window functions against a SQLite sandbox.
* `04_pyspark_concepts.ipynb` — PySpark concepts, guarded so it runs without Spark installed.
* `05_interview_questions.ipynb` — a mixed question bank for mock interviews.